In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from collections import Counter
import pandas as pd
import numpy as np
import random

In [3]:
!pip install datasets sentencepiece

In [4]:
from datasets import load_dataset
import pandas as pd

In [5]:
from google.colab import files

uploaded = files.upload()

Saving fra.txt to fra.txt


In [6]:
import pandas as pd

df = pd.read_csv(
    "/content/fra.txt",
    sep="\t",
    header=None,
    names=["English", "French", "License"],
    usecols=[0, 1]
)

print(df.head())
print(df.shape)

  English      French
0     Go.        Va !
1     Go.     Marche.
2     Go.  En route !
3     Go.     Bouge !
4     Hi.     Salut !
(240521, 2)


In [7]:
# Keep only the first 60,000 sentence pairs
df = df.iloc[:60000].copy()

print("Dataset size:", len(df))

Dataset size: 60000


In [8]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'fra.txt', 'sample_data']


In [9]:
sentence = "I love deep learning"
print(type(sentence))
def tokenise(sentence):
  return sentence.lower().split()

<class 'str'>


In [10]:
from collections import Counter

def build_vocab(sentences):

    counter = Counter()

    # Count all words
    for sentence in sentences:
        words = tokenise(sentence)
        counter.update(words)

    # Create vocabulary only once
    vocab = {
        "<PAD>": 0,
        "<SOS>": 1,
        "<EOS>": 2,
        "<UNK>": 3
    }

    # Add all unique words
    for word in counter:
        vocab[word] = len(vocab)

    return vocab

In [11]:
def numericalisation(sentence,vocab):
  tokens=tokenise(sentence)
  ids=[vocab["<SOS>"]]
  for token in tokens:
    ids.append(vocab.get(token, vocab["<UNK>"]))
  ids.append(vocab["<EOS>"])
  return ids

In [12]:
def pad_sequence(sequence, max_length, pad_idx):
    sequence = sequence[:max_length]
    while len(sequence) < max_length:
        sequence.append(pad_idx)
    return sequence

In [13]:
class translationDataset(Dataset):
  def __init__(self, dataframe, src_vocab, trg_vocab, max_src_len,max_trg_len ):
    self.dataframe=dataframe
    self.src_vocab=src_vocab
    self.trg_vocab=trg_vocab
    self.max_src_len=max_src_len
    self.max_trg_len=max_trg_len
  def __len__(self):
      return len(self.dataframe)
  def __getitem__(self,idx):
    src_sentence = self.dataframe.iloc[idx]["English"]
    trg_sentence = self.dataframe.iloc[idx]["French"]

    src_tokens = tokenise(src_sentence)
    trg_tokens = tokenise(trg_sentence)

    src_ids = [self.src_vocab.get(token, self.src_vocab["<UNK>"]) for token in src_tokens]
    trg_ids = [self.trg_vocab.get(token, self.trg_vocab["<UNK>"]) for token in trg_tokens]

    src_ids = [self.src_vocab["<SOS>"]] + src_ids + [self.src_vocab["<EOS>"]]
    trg_ids = [self.trg_vocab["<SOS>"]] + trg_ids + [self.trg_vocab["<EOS>"]]

    src_ids = src_ids[:self.max_src_len]
    src_ids += [self.src_vocab["<PAD>"]] * (self.max_src_len - len(src_ids))
    trg_ids = trg_ids[:self.max_trg_len]
    trg_ids += [self.trg_vocab["<PAD>"]] * (self.max_trg_len - len(trg_ids))

    src_tensor = torch.tensor(src_ids, dtype=torch.long)
    trg_tensor = torch.tensor(trg_ids, dtype=torch.long)

    return src_tensor, trg_tensor
    row = self.dataframe.iloc[idx]

    src_sentence = row["English"]
    trg_sentence = row["French"]

    src_tokens = tokenise(src_sentence)
    trg_tokens = tokenise(trg_sentence)

    src_ids = []

    for token in src_tokens:
      id = self.src_vocab.get(token, self.src_vocab["<UNK>"])
      src_ids.append(id)
    trg_ids = []

    for token in trg_tokens:
      id = self.trg_vocab.get(token, self.trg_vocab["<UNK>"])
      trg_ids.append(id)
      # Add Start Of Sentence and End Of Sentence tokens
    src_ids = [self.src_vocab["<SOS>"]] + src_ids + [self.src_vocab["<EOS>"]]
    trg_ids = [self.trg_vocab["<SOS>"]] + trg_ids + [self.trg_vocab["<EOS>"]]
    # Truncate sequences if they exceed maximum length
    src_ids = src_ids[:self.max_src_len]
    trg_ids = trg_ids[:self.max_trg_len]
    # Pad source sequence
    src_ids += [self.src_vocab["<PAD>"]] * (self.max_src_len - len(src_ids))

    # Pad target sequence
    trg_ids += [self.trg_vocab["<PAD>"]] * (self.max_trg_len - len(trg_ids))

    src_tensor = torch.tensor(src_ids, dtype=torch.long)
    trg_tensor = torch.tensor(trg_ids, dtype=torch.long)
    return src_tensor, trg_tensor








In [14]:
src_vocab = build_vocab(df["English"])
trg_vocab = build_vocab(df["French"])

In [15]:
import torch.nn as nn

src_embedding = nn.Embedding(
    num_embeddings=len(src_vocab),
    embedding_dim=512
)

In [16]:
train_dataset = translationDataset(
    dataframe=df,
    src_vocab=src_vocab,
    trg_vocab=trg_vocab,
    max_src_len=20,
    max_trg_len=20
)

In [17]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,

    shuffle=True
)

In [18]:
import math
import torch
import torch.nn as nn
class InputEmbedding(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)

    self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


In [19]:
import torch
import torch.nn as nn
class Encoder(nn.Module):
  def __init__(self,
             input_vocab_size,
             embed_dim=256,
             hidden_dim=512,
             n_layers=1,
             dropout=0.3,
             pad_idx=0):
    super().__init__()
    self.embedding = nn.Embedding(
    input_vocab_size,
    embed_dim,
    padding_idx=pad_idx
)
    self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
    self.dropout = nn.Dropout(dropout)
  def forward(self, src):
    embedded = self.dropout(self.embedding(src))
    outputs, (hidden, cell) = self.lstm(embedded)
        # hidden, cell: (n_layers, batch, hidden_dim) -- this IS the context
    return hidden, cell





In [20]:
class Decoder(nn.Module):
  def __init__(self, output_vocab_size, embed_dim=256, hidden_dim=512,
                 n_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(output_vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        self.fc_out = nn.Linear(hidden_dim, output_vocab_size)
        self.dropout = nn.Dropout(dropout)

  def forward(self, input_token, hidden, cell):
     input_token = input_token.unsqueeze(1)                 # (batch, 1)
     embedded = self.dropout(self.embedding(input_token))
     output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
     prediction = self.fc_out(output.squeeze(1))            # (batch, output_vocab_size)
     return prediction, hidden, cell









In [21]:
import random

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
       batch_size, trg_len = trg.shape
       trg_vocab_size = self.decoder.fc_out.out_features

       outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=self.device)

       hidden, cell = self.encoder(src)

       input_token = trg[:, 0]

       for t in range(1, trg_len):

        prediction, hidden, cell = self.decoder(input_token, hidden, cell)

        outputs[:, t, :] = prediction

        teacher_force = random.random() < teacher_forcing_ratio
        top1 = prediction.argmax(1)


        input_token = trg[:, t] if teacher_force else top1



    # <-- IMPORTANT: outside the loop
       return outputs





In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_DIM  = len(src_vocab)
OUTPUT_DIM = len(trg_vocab)
EMBED_DIM  = 256
HIDDEN_DIM = 512
N_LAYERS   = 2
DROPOUT    = 0.3
PAD_IDX    = src_vocab["<PAD>"]   # 0, same index in both vocabs

encoder = Encoder(INPUT_DIM, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_IDX)
decoder = Decoder(OUTPUT_DIM, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_IDX)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab["<PAD>"])

In [23]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [24]:
def train_one_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0

    print("Starting epoch...")

    for batch_idx, (src, trg) in enumerate(loader):
        print(f"Batch {batch_idx}")

        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()

        print("Forward...")
        output = model(src, trg, teacher_forcing_ratio=0.5)

        print("Loss...")
        output_dim = output.shape[-1]
        output = output[:, 1:, :].reshape(-1, output_dim)
        trg_flat = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg_flat)

        print("Backward...")
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

        print("Done batch", batch_idx)

    return epoch_loss / len(loader)




In [25]:
def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=20):
    model.eval()

    tokens = tokenise(sentence)
    ids = [src_vocab["<SOS>"]] + [src_vocab.get(t, src_vocab["<UNK>"]) for t in tokens] + [src_vocab["<EOS>"]]
    ids = pad_sequence(ids, max_len, src_vocab["<PAD>"])
    src_tensor = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)  # (1, max_len)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    trg_ids = [trg_vocab["<SOS>"]]
    inv_trg_vocab = {idx: word for word, idx in trg_vocab.items()}

    for _ in range(max_len):
        input_token = torch.tensor([trg_ids[-1]], device=device)
        with torch.no_grad():
            prediction, hidden, cell = model.decoder(input_token, hidden, cell)
        next_id = prediction.argmax(1).item()
        if next_id == trg_vocab["<EOS>"]:
            break
        trg_ids.append(next_id)

    words = [inv_trg_vocab[i] for i in trg_ids[1:]]   # drop <SOS>
    return " ".join(words)

In [26]:
N_EPOCHS = 10

for epoch in range(N_EPOCHS):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}")

Streaming output truncated to the last 5000 lines.
Forward...
Loss...
Backward...
Done batch 875
Batch 876
Forward...
Loss...
Backward...
Done batch 876
Batch 877
Forward...
Loss...
Backward...
Done batch 877
Batch 878
Forward...
Loss...
Backward...
Done batch 878
Batch 879
Forward...
Loss...
Backward...
Done batch 879
Batch 880
Forward...
Loss...
Backward...
Done batch 880
Batch 881
Forward...
Loss...
Backward...
Done batch 881
Batch 882
Forward...
Loss...
Backward...
Done batch 882
Batch 883
Forward...
Loss...
Backward...
Done batch 883
Batch 884
Forward...
Loss...
Backward...
Done batch 884
Batch 885
Forward...
Loss...
Backward...
Done batch 885
Batch 886
Forward...
Loss...
Backward...
Done batch 886
Batch 887
Forward...
Loss...
Backward...
Done batch 887
Batch 888
Forward...
Loss...
Backward...
Done batch 888
Batch 889
Forward...
Loss...
Backward...
Done batch 889
Batch 890
Forward...
Loss...
Backward...
Done batch 890
Batch 891
Forward...
Loss...
Backward...
Done batch 891
Batch 8

In [27]:
torch.save(model.state_dict(), "seq2seq_lstm.pth")

In [29]:
test_sentences = [

    "how are you",
    "good morning",
    "thank you",
    "where are you",
]

for sentence in test_sentences:
    translation = translate_sentence(
        sentence,
        src_vocab,
        trg_vocab,
        model,
        device
    )

    print(f"English : {sentence}")
    print(f"French  : {translation}")
    print("-" * 40)

English : how are you
French  : comment vas-tu ?
----------------------------------------
English : good morning
French  : bonjour !
----------------------------------------
English : thank you
French  : merci !
----------------------------------------
English : where are you
French  : où êtes-vous ?
----------------------------------------
